In [ ]:
import datetime
import numpy as np
import pandas as pd
import scipy
import matplotlib as mpl
import matplotlib.pyplot as plt
import mplfinance as mpf

In [ ]:
%matplotlib notebook
%config InlineBackend.figure_format = 'retina'
np.set_printoptions(precision=6, suppress=True)

In [ ]:
filename = r'PLTR_20241006_113704_H.csv'
df = pd.read_csv(filename, index_col=0, parse_dates=['date']).drop(columns=['average'])
df.info()

In [ ]:
%matplotlib widget

In [ ]:
df.plot(y='close', title='PLTR', figsize=(10, 6))

In [ ]:
df[ pd.isna(df.close) ]

In [ ]:
# df['t'] = df.index.to_series().diff().dt.total_seconds().div(60).fillna(0)
# df['t'] = df.groupby(df.index.date)['t'].cumsum()


In [ ]:
df.head()

In [ ]:
#open_close_list = [(row['open'], row['close']) for _, row in df.iloc[-10:].iterrows()]
np.array([(row['open'], row['close']) for _, row in df.iloc[-10:].iterrows()]).flatten()
#open_close_list

In [ ]:
from ib_insync.objects import BarData
from collections import defaultdict
from datetime import timedelta

def resample_bars_to_list(bars, resample_interval='2T'):
    """
    Resample a list of BarData objects into a specified interval and return a list of resampled BarData.

    Parameters:
    bars (list of BarData): The list of BarData objects to resample.
    resample_interval (str): The resampling interval (e.g., '2T' for 2 minutes).

    Returns:
    list of BarData: The resampled OHLC data.
    """
    interval_seconds = pd.to_timedelta(resample_interval).total_seconds()
    resampled_bars = []
    current_interval_start = bars[0].time
    current_interval_end = current_interval_start + timedelta(seconds=interval_seconds)
    
    current_data = defaultdict(list)
    
    for bar in bars:
        if bar.time >= current_interval_end:
            if current_data:
                resampled_bars.append(BarData(
                    time=current_interval_start,
                    open=current_data['open'][0],
                    high=max(current_data['high']),
                    low=min(current_data['low']),
                    close=current_data['close'][-1],
                    volume=sum(current_data['volume']),
                    barCount=sum(current_data['barCount'])
                ))
            current_interval_start = current_interval_end
            current_interval_end = current_interval_start + timedelta(seconds=interval_seconds)
            current_data = defaultdict(list)
        
        current_data['time'].append(bar.time)
        current_data['open'].append(bar.open)
        current_data['high'].append(bar.high)
        current_data['low'].append(bar.low)
        current_data['close'].append(bar.close)
        current_data['volume'].append(bar.volume)
        current_data['barCount'].append(bar.barCount)
    
    if current_data:
        resampled_bars.append(BarData(
            time=current_interval_start,
            open=current_data['open'][0],
            high=max(current_data['high']),
            low=min(current_data['low']),
            close=current_data['close'][-1],
            volume=sum(current_data['volume']),
            barCount=sum(current_data['barCount'])
        ))
    
    return resampled_bars

# # Example usage
# resampled_bars_list = resample_bars_to_list(bars)
# for bar in resampled_bars_list:
#     print(bar)
    
def resample_bars(bars, resample_interval='2T'):
    """
    Resample a list of BarData objects into a specified interval.

    Parameters:
    bars (list of BarData): The list of BarData objects to resample.
    resample_interval (str): The resampling interval (e.g., '2T' for 2 minutes).

    Returns:
    pd.DataFrame: The resampled OHLC data.
    """
    # Convert the list of BarData objects to a DataFrame
    data = {
        'time': [bar.time for bar in bars],
        'open': [bar.open for bar in bars],
        'high': [bar.high for bar in bars],
        'low': [bar.low for bar in bars],
        'close': [bar.close for bar in bars],
        'volume': [bar.volume for bar in bars],
        'barCount': [bar.barCount for bar in bars]
    }
    df = pd.DataFrame(data)
    df.set_index('time', inplace=True)

    # Resample the DataFrame
    resampled_df = df.resample(resample_interval).agg({
        'open': 'first',
        'high': 'max',
        'low': 'min',
        'close': 'last',
        'volume': 'sum',
        'barCount': 'sum'
    })

    return resampled_df

# # Example usage
# bars = [
#     BarData(time=pd.Timestamp('2024-09-12 09:30:00'), open=224.66, high=225.90, low=224.01, close=224.62, volume=646910, barCount=1830),
#     BarData(time=pd.Timestamp('2024-09-12 09:31:00'), open=224.54, high=225.84, low=223.83, close=225.49, volume=430284, barCount=1835),
#     # Add more BarData objects here...
# ]

# resampled_bars = resample_bars(bars)
# print(resampled_bars)

In [ ]:
how_agg = {
    'open': 'first',
    'close': 'last',
    'high': 'max',
    'low': 'min',
    'volume': 'sum',
    'barCount': 'sum',
    # 'n': 'count'
}
expected_n = 5

In [ ]:
# df5m = df.resample('5min', on='date').agg(how_agg).reset_index(drop=False)
df5m = df.set_index('date').resample('5min').agg(how_agg).reset_index(drop=False).dropna().reset_index(drop=True)

df5m.info()

In [ ]:
df5m

In [ ]:
df5m['h-l'] = df5m['high'] - df5m['low']
df5m['c-o'] = df5m['close'] - df5m['open']

In [ ]:
from IPython.display import display, Javascript 
    
def auto_scroll(): 
    display(Javascript(''' 
        var output_area = document.querySelector('.output_area'); 
        output_area.scrollTop = output_area.scrollHeight; 
    ''')) 
    
# Call the function after your long output 
auto_scroll() 

In [ ]:
# plot df5m['h-l'] vs df5m['date']
fig, axs = plt.subplots(2,1) #,figsize=(100, 6)

axs[0].bar(df5m.index, df5m['h-l'])
axs[0].set_ylabel('High$-$Low')
# axs[0].set_ylim(0, 0.8)
axs[1].bar(df5m.index, df5m['c-o'])
# axs[1].set_ylim(-0.8, 0.8)
axs[1].set_ylabel('Close$-$Open')
# ax.xaxis.set_major_locator(mpl.dates.WeekdayLocator(byweekday=mpl.dates.MO))
# ax.xaxis.set_minor_locator(mpl.dates.DayLocator())
# ax.xaxis.set_major_formatter(mpl.dates.DateFormatter('%Y-%m-%d'))
# fig.autofmt_xdate()
# fig.tight_layout()
plt.show()

In [ ]:
df5m['c-o'].argmin(), df5m['c-o'].argmax()

In [ ]:
df5m['h-l'].argmin(), df5m['h-l'].argmax(), df5m['h-l'].min(), df5m['h-l'].max()

In [ ]:
axs[1].get_ymargin()

In [ ]:

df5m['logret']=np.log(df5m['close'] / df5m['close'].shift(1)).combine_first(np.log(df5m['close'] / df5m['open']))
df5m['logret_hl']=np.log(df5m['high'] / df5m['low'])
df5m


In [ ]:
df1h = df.resample('1h', on='date', origin='start').agg({'open': 'first', 'high': 'max', 'low': 'min', 'close': 'last', 'volume': 'sum', 'barCount': 'sum'})
df1h['logret'] = np.log(df1h['close'] / df1h['close'].shift(1)).combine_first(np.log(df1h['close'] / df1h['open']))
df1h

In [ ]:
s = slice(0, None)
y = [b for b in df5m['close'].iloc[s]]
x = np.arange(len(y))

In [ ]:
price_func = scipy.interpolate.PchipInterpolator(x, y, extrapolate=False)
price_func_deriv = price_func.derivative()
price_slopes = price_func_deriv(x)
price_slopes